# Keeping the last reading

**Temporal-stage notebook 1 in `notebooks/temporal/`.** `notebooks/README.md` says how every notebook is laid out, and `notebooks/temporal/README.md` says what the notebooks in this folder share: the window, the sensor, how the cost of a stage is counted, and which stored runs they read.

## What this notebook is for

The network begins with a **temporal stage**. For each of the 14 sensors it receives the readings of the last 30 flight cycles and decides what that sensor's node carries into the graph layers. The stage in this notebook, `last_cycle`, keeps the newest of the 30 readings and drops the other 29.

It is this project's default, and this project's choice (assumption A-07). The paper's Table 4 gives each sensor one number entering the first graph layer, and its Section 2.2 makes that number with a recurrent network for each sensor. This project leaves that network out by default (assumption A-11), and the newest reading takes its place.

The notebook runs the stage on one real window, shows what it keeps and what it drops, counts what it costs, and lists the stored five-seed runs that used it.

## What it does not do

- **It does not train anything.** The stage has nothing to learn, and the network's results are read from stored runs.
- **It does not compare this stage with another.** `07_comparison_fd001.ipynb` does that, with paired tests and with the whole configuration named.
- **It does not show that the newest reading is enough.** It shows what is thrown away.

## The notebook in one list

| # | Step | What goes in | What comes out |
|---|------|--------------|----------------|
| 1 | **Load the window** | `config/fd001.yaml`, seed 42 | the held-back engine's last window, 14 sensors by 30 cycles |
| 2 | **Run the stage** | the window | one number per sensor |
| 3 | **Look at what is dropped** | T50's 29 earlier readings, and every held-back window | two real windows the stage turns into the same number for T50 |
| 4 | **Count the cost** | the network built with this stage | learned numbers in the stage and in the whole network |
| 5 | **Read the stored results** | `reports/` | every stored five-seed run of this stage under this folder's settings |

## 1. Background

### What enters the stage, and what leaves it

One example the network reads is a **window**: the 14 sensors, each with its readings from the last 30 flight cycles. As an array it has the shape `(14, 30)`, one row per sensor and one column per cycle, oldest first. Training and prediction hand the network several windows at once, a **batch**, so the stage receives an array of shape `(batch, 14, 30)`.

Every temporal stage returns `(batch, 14, k)`: `k` numbers for each sensor. The graph layers after it never see a cycle. They see those `k` numbers per sensor and nothing else, so whatever the stage drops is gone for the rest of the network.

```
(batch, 14 sensors, 30 cycles)  →  last_cycle  →  (batch, 14 sensors, 1)
```

For this stage `k = 1`, and for one sensor the whole stage is one operation: take entry 30 of 30. In the code it is `windows[:, :, -1:]` — every window, every sensor, and the last cycle only. It is written as the slice `-1:` rather than the index `-1` so that the third axis survives with length 1 instead of disappearing: the result is `(batch, 14, 1)`, not `(batch, 14)`.

### Why one number per sensor

The paper's Table 4 (its page 7) gives the input of the first graph layer as `[batch size,14,1]` and that layer's weight matrix as `W(1) ∈ ℝ(1×32)`: the layer multiplies each sensor's one number by 32 learned numbers to make 32. The table fixes how many numbers the temporal stage must hand on. It does not say what the one number is. The text under Equation 8 (page 6) says the node features come "from LSTM networks", the recurrent networks of the paper's Section 2.2. Without them something else has to fill the slot, and this stage fills it with the newest reading.

**Do not confuse** the window's length with the stage's output width. The window is 30 cycles long whichever stage is used; `k` is how many numbers per sensor survive the stage.

**Do not confuse** dropping 29 readings with dropping 29 cycles of history. Each reading in the window was smoothed before the window was cut (assumptions A-04 and A-09): the smoothed value at a cycle is 0.5 times that cycle's raw reading, plus 0.25 times the one before, plus 0.125 times the one before that, and so on. So the newest reading already blends the engine's recent raw readings, the last six of them carrying 0.5 + 0.25 + 0.125 + 0.0625 + 0.03125 + 0.015625 = 0.984375 of its weight. What the stage drops is the 29 earlier *smoothed* values, and with them any description of where the sensor sat and which way it moved across the 30 cycles.

## 2. Setup

In [ ]:
from __future__ import annotations

import logging
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display

# The project's top folder is the nearest one above this notebook that holds
# pyproject.toml. Adding its src/ folder to Python's search path makes the
# package importable from a clone that was never installed.
PROJECT_ROOT = next(
    folder for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "pyproject.toml").is_file()
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

try:
    import torch
except ImportError as missing:
    raise ImportError(
        "This notebook builds the network's temporal stage, which needs PyTorch, and PyTorch is "
        'not installed. Install the project with  pip install -e ".[model]"'
    ) from missing

from thesis_gnn_rul_xai.config import load_config
from thesis_gnn_rul_xai.data.pipeline import run_preprocessing
from thesis_gnn_rul_xai.models import ModelConfig, build_model
from thesis_gnn_rul_xai.models.temporal import LastCycle
from thesis_gnn_rul_xai.training import TrainingConfig, find_runs
from thesis_gnn_rul_xai.utils.logging_setup import configure_logging

configure_logging(logging.ERROR)
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", None)    # a run's folder name is printed whole
plt.rcParams.update({"figure.dpi": 110, "font.size": 9})

print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch     : {torch.__version__}")

## 3. The window

### Step 1 — Load the window

**What.** Rebuild the data from `config/fd001.yaml`, take the held-back engine with the most windows, and take its last window. Print the 30 readings of T50, the temperature at the outlet of the low-pressure turbine, the sensor every notebook in this folder follows.

**Why.** Every notebook here works on this same window, so a difference between two notebooks is a difference between two stages. `notebooks/temporal/README.md` says why this engine, this window and this sensor, and what the units of a reading are.

**Check.** The window must be 14 sensors by 30 cycles, come from an engine the training never saw, and end at the cycle the engine failed.

In [ ]:
config = load_config(PROJECT_ROOT / "config" / "fd001.yaml")
prepared = run_preprocessing(config)

sensors = list(prepared.sensor_symbols)

# The held-back engine with the most windows, and its last window.
held_back = prepared.validation
units, counts = np.unique(held_back.units, return_counts=True)
ENGINE = int(units[np.argmax(counts)])
engine_rows = np.flatnonzero(held_back.units == ENGINE)
engine_rows = engine_rows[np.argsort(held_back.end_cycles[engine_rows])]

SENSOR = "T50"
s = sensors.index(SENSOR)
window = held_back.features[engine_rows[-1]]             # (14 sensors, 30 cycles)
end_cycle = int(held_back.end_cycles[engine_rows[-1]])
cycles = np.arange(end_cycle - window.shape[1] + 1, end_cycle + 1)
readings = window[s]                                     # T50's 30 readings, oldest first

print(f"Engine {ENGINE}: {len(engine_rows)} windows, held back from training under seed {config.seed}.")
print(f"Worked window: cycles {cycles[0]} to {cycles[-1]}, shape {window.shape}, "
      f"remaining life at its end {held_back.targets[engine_rows[-1]]:.0f} cycles.\n")
display(pd.DataFrame([readings], index=[f"{SENSOR} reading"],
                     columns=pd.MultiIndex.from_arrays([range(1, 31), cycles],
                                                       names=["position", "engine cycle"])).round(6))

window_checks = {
    "the window is 14 sensors by 30 cycles": window.shape == (14, 30),
    "its engine is not a training engine": ENGINE not in set(prepared.train.units),
    "it ends at the cycle the engine failed": float(held_back.targets[engine_rows[-1]]) == 0.0,
}
assert all(window_checks.values()), window_checks
print("OK  " + "; ".join(window_checks))

## 4. The stage, step by step

### Step 2 — Run the stage

**What.** Hand the window to `LastCycle(30)` as a batch of one, and print what each sensor's node receives.

**Why.** The number checked is then the stage's own output, not a reading of the table above.

**Check.** For every sensor the output must be exactly the window's last column.

In [ ]:
stage = LastCycle(window_size=30)
with torch.no_grad():
    handed_on = stage(torch.from_numpy(window[None]))      # (1, 14, 1)

print(f"in  {tuple(window[None].shape)}    out {tuple(handed_on.shape)}\n")
display(pd.DataFrame({"handed on": handed_on[0, :, 0].numpy()}, index=sensors).T.round(6))

stage_keeps_last_column = bool(np.array_equal(handed_on[0, :, 0].numpy(), window[:, -1]))
assert stage_keeps_last_column
print(f"OK  each number is that sensor's reading at cycle {end_cycle}; "
      f"{SENSOR}'s is {float(handed_on[0, s, 0]):.6f}.")

### Step 3 — What the other 29 readings held

**What.** Two looks at what the stage drops. First, T50's 29 earlier readings in this window, drawn beside the one kept. Second, a search through every held-back window for the two whose newest T50 readings are within 0.001 of each other while the averages of their 30 readings are as far apart as possible.

**Why.** This stage costs nothing in learned numbers. Its cost is information: anything in the 29 earlier readings — where the sensor sat, which way it was moving — never reaches the graph. The search makes that concrete with real readings: two windows whose T50 histories differ, which the stage turns into practically the same number for T50's node.

**Check.** The stage's outputs for T50 on the two windows must differ by less than 0.001.

In [ ]:
held_newest = held_back.features[:, s, -1]
held_average = held_back.features[:, s, :].mean(axis=1)

# Walk through the windows in order of their newest reading; for each, look only
# at the windows whose newest reading is less than 0.001 higher.
order = np.argsort(held_newest)
widest, first, second = 0.0, None, None
for left in range(len(order)):
    right = left + 1
    while right < len(order) and held_newest[order[right]] - held_newest[order[left]] < 0.001:
        gap = abs(held_average[order[right]] - held_average[order[left]])
        if gap > widest:
            widest, first, second = gap, int(order[left]), int(order[right])
        right += 1

pair = pd.DataFrame({
    "engine": held_back.units[[first, second]],
    "window ends at cycle": held_back.end_cycles[[first, second]],
    "remaining life (cycles)": held_back.targets[[first, second]],
    f"{SENSOR} newest reading": held_newest[[first, second]],
    f"{SENSOR} average of its 30": held_average[[first, second]],
}, index=["window A", "window B"])
print(f"Searched {len(held_newest):,} held-back windows.")
display(pair.round(4))

with torch.no_grad():
    pair_out = stage(torch.from_numpy(held_back.features[[first, second]]))[:, s, 0].numpy()
same_number_for_t50 = bool(abs(pair_out[0] - pair_out[1]) < 1e-3)
assert same_number_for_t50
print(f"OK  the stage hands T50's node {pair_out[0]:.4f} and {pair_out[1]:.4f}.")

figure, (left, right) = plt.subplots(1, 2, figsize=(11, 3.4))
left.plot(range(1, 31), readings, "-", color="0.8", linewidth=1)
left.plot(range(1, 30), readings[:-1], "o", color="0.55", markersize=4, label="dropped (29 readings)")
left.plot([30], readings[-1:], "o", color="C3", markersize=8, label="kept")
left.set(title=f"Engine {ENGINE}, cycles {cycles[0]} to {cycles[-1]}", xlabel="position in the window",
         ylabel=f"{SENSOR} reading")
left.legend()
for row, label, colour in ((first, "A", "C0"), (second, "B", "C1")):
    right.plot(range(1, 31), held_back.features[row, s], "o-", markersize=3, color=colour,
               label=f"window {label}: engine {held_back.units[row]}, "
                     f"{held_back.targets[row]:.0f} cycles left")
right.set(title="Two windows the stage cannot tell apart at T50", xlabel="position in the window",
          ylabel=f"{SENSOR} reading")
right.legend()
figure.tight_layout()
plt.show()

#### Reading the figure

**On the left**, the stage keeps 2.740, the reading at cycle 269, and drops the 29 before it, which run from 1.250 to 2.828. The kept reading is not even the window's largest: cycle 268's is. That T50 climbed by more than one standard deviation across these 30 cycles reaches the graph only in so far as the newest reading happens to be high.

**On the right**, two windows from engines 43 and 26, both early in life, ending at cycles 38 and 34. Their newest T50 readings are −0.9282 and −0.9280, so T50's node receives practically the same number from both. Their histories are not alike: window A's readings sit below its newest reading for most of the window and average −1.3028, while window B's sit above it, average −0.0655, and fall to it at the end.

Both windows have 125 cycles of remaining life, the ceiling every label is capped at (assumption A-01), so this pair shows what the stage loses, not a mistake it causes. The network still sees the other 13 sensors, and whether T50's history would have helped is measured only by training the network with stages that keep it, in the stored runs `07_comparison_fd001.ipynb` reads.

### Step 4 — What it costs

**What.** Build the whole network with this stage on the paper's sensor graph, the way `scripts/train.py` builds it, and count the learned numbers in each part. A **learned number** is one the training adjusts — a multiplier or an offset.

**Why.** A stage costs two things: what it drops, which step 3 showed, and what it adds to the network's size. This stage adds nothing.

**Check.** The stage must hold no learned numbers, the first graph layer's weight matrix must have the shape Table 4 prints, `(1, 32)`, and the whole network must hold 61,729 numbers.

In [ ]:
TABLE_4_NAMES = {"graph_layers.0": "GCN-1", "graph_layers.1": "GCN-2",
                 "graph_layers.2": "GCN-3", "fc1": "FC-1", "fc2": "FC-2"}


def parameter_table(network):
    """Learned numbers in each part of a network, the temporal stage first."""
    parts = {"temporal stage": 0}
    for name, parameter in network.named_parameters():
        part = "temporal stage" if name.startswith("temporal.") else TABLE_4_NAMES[name.rsplit(".", 1)[0]]
        parts[part] = parts.get(part, 0) + parameter.numel()
    parts["whole network"] = sum(parts.values())
    return pd.DataFrame({"learned numbers": parts})


network = build_model(prepared.adjacency_normalized, ModelConfig(temporal="last_cycle"),
                      window_size=config.window_size, rul_max=config.rul_max)
costs = parameter_table(network)
display(costs.T)

first_weight = tuple(network.graph_layers[0].weight.shape)
cost_checks = {
    "the stage learns nothing": int(costs.loc["temporal stage", "learned numbers"]) == 0,
    "W(1) is (1, 32)": first_weight == (1, 32),
    "the whole network holds 61,729": int(costs.loc["whole network", "learned numbers"]) == 61_729,
}
assert all(cost_checks.values()), cost_checks
print("OK  " + "; ".join(cost_checks))

The five weight matrices Table 4 prints come to 32 + 2,048 + 2,048 + 57,344 + 128 = 61,600 numbers. The other 129 are the two offsets of the fully connected layers, 128 for FC-1 and 1 for FC-2, which Table 4 does not list but Equations 12 and 13 (page 6) print. The graph layers carry no offset, because Equations 8 to 10 print none (assumption A-12). So 61,729 is derived from three places in the paper and one choice of this project, not read off a table.

### Step 5 — The stored results of this stage

**What.** Every stored five-seed run that used this stage under this folder's settings, found with `find_runs`, the function in `src/thesis_gnn_rul_xai/training/stored.py` that reads `reports/`, in the top of `reports/` and in the sensitivity study's folder. For each run the table gives the mean and standard deviation over its five seeds of the root mean squared error (RMSE), in cycles, on the 100 test engines and on the held-back engines; notebook 02's step 11 defines it. Runs that saved only part of their settings — the oldest saved nothing about their training but the learning-rate path — are listed with what they did not save, and their seeds are set beside those of the runs that saved everything.

**Why.** A stage cannot produce a result on its own. An error in cycles belongs to the whole network, trained, and to the settings it was trained under, so the cell prints those settings beside the runs.

In [ ]:
STUDY = PROJECT_ROOT / "reports" / "sensitivity_fd65a3b-dirty_20260929T101059Z" / "runs" / "FD001"
DEFAULT_TRAINING = TrainingConfig()
TRAINING_FIELDS = ("epochs", "batch_size", "learning_rate", "final_learning_rate",
                   "early_stopping_patience", "lr_schedule")
WANTED = [ModelConfig(temporal="last_cycle")]


def training_settings(run) -> str:
    """'default' or 'other' from the training settings a run saved; 'not saved' if it saved only the learning-rate path."""
    recorded = run.summary.get("training")
    if recorded is None:
        return "not saved" if run.summary.get("lr_schedule") == DEFAULT_TRAINING.lr_schedule else "other"
    default = all(recorded.get(field) == getattr(DEFAULT_TRAINING, field) for field in TRAINING_FIELDS)
    return "default" if default else "other"


def not_saved(run) -> str:
    """The settings a run did not save, or a dash if it saved them all."""
    missing = [] if run.summary.get("training") else ["training settings"]
    if not (run.path / "model_config.yaml").is_file():
        missing.append("network sizes" if "temporal_stage" in run.summary else "network sizes, stage")
    return ", ".join(missing) or "—"


def machine(run) -> str:
    """The device a run's first seed asked for, and the PyTorch build the run recorded."""
    saved = run.path / f"seed_{run.seeds[0]}" / "training_config.yaml"
    device = yaml.safe_load(saved.read_text(encoding="utf-8")).get("device") if saved.is_file() else None
    build = ((run.summary.get("environment") or {}).get("packages") or {}).get("torch")
    return f"{device or 'not recorded'}, PyTorch {build or 'not recorded'}"


stored = [
    run
    for root in (None, STUDY)
    for run in find_runs(root=root, preprocessing=config, seeds=5, split="varies_with_seed",
                         early_stopping=True)
    if run.model_config in WANTED and training_settings(run) != "other"
]
print("Settings every run below shares, where it saved them:")
print("  preprocessing:", {k: v for k, v in config.to_dict().items() if k != "notes"})
print("  training:     ", {field: getattr(DEFAULT_TRAINING, field) for field in TRAINING_FIELDS},
      "and the pass that scored best on the held-back engines restored")
for model_config in WANTED:
    print("  network:      ", model_config.to_dict())
print()
if stored:
    display(pd.DataFrame([{
        "run, under reports/": run.path.resolve().relative_to((PROJECT_ROOT / "reports").resolve()).as_posix(),
        "commit": run.summary["commit"],
        **({"one network for all sensors": run.model_config.shared_across_sensors}
           if len(WANTED) > 1 else {}),
        "device asked for, PyTorch build": machine(run),
        "not saved by the run": not_saved(run),
        "seeds": ", ".join(map(str, run.seeds)),
        "test RMSE": f"{run.results['rmse'].mean():.3f} ± {run.results['rmse'].std():.3f}",
        "held-back RMSE": f"{run.results['validation_rmse'].mean():.3f} ± "
                          f"{run.results['validation_rmse'].std():.3f}",
    } for run in stored]).set_index("run, under reports/"))
else:
    print("reports/ holds no five-seed run of this stage under these settings. Make one with\n\n"
          "    python scripts/train.py --config config/fd001.yaml --seeds 5\n")

# A run that saved less is set beside the runs that saved everything, seed by seed.
complete = [run for run in stored if not_saved(run) == "—"]
repeats = {}
for run in stored:
    if not_saved(run) == "—":
        continue
    mine = run.results.set_index("seed").sort_index()[["rmse", "validation_rmse"]].to_numpy()
    repeats[run.name] = [
        other.summary["commit"] for other in complete
        if np.array_equal(other.results.set_index("seed").sort_index()[["rmse", "validation_rmse"]].to_numpy(), mine)
    ]
    unsaid = not_saved(run).split(", ")
    unsaid = unsaid[0] if len(unsaid) == 1 else ", ".join(unsaid[:-1]) + " or " + unsaid[-1]
    twins = repeats[run.name]
    print(f"The run at {run.summary['commit']} saved no {unsaid}; its five seeds give the same test and held-back "
          f"RMSE, to every stored digit, as the run{'s' if len(twins) > 1 else ''} at {', '.join(twins) or 'no commit'}.")
unsaved_runs_repeat = all(repeats.values())

**Which published row these may be set beside.** These runs have no recurrent stage, so the published row they may be set beside is the paper's "ESTGNN w/o LSTM", 14.93 RMSE on FD001 (its Table 11, page 12), never the full model's 11.78. Even beside that row they are not a reproduction of it. The paper does not say what its variant hands the graph in place of the recurrent output. These runs hold back a fifth of the training engines and restore the pass over the training data that scored best on them, neither of which the paper describes (assumptions A-06 and A-18). And their test answers are capped at 125 cycles, which can only lower the RMSE and the NASA score (assumption A-02).

**Seven runs, two results.** Six runs — at `9856794-dirty`, `ae59bc1-dirty`, `f82c76c-dirty`, `f386291-dirty`, `5602773-dirty` and `f4f40ad` — record the same mean and spread, on the test engines and on the held-back ones. The three oldest saved nothing about their training but the learning-rate path, and no network sizes, and the two oldest did not save their stage either; the last lines of the cell show that each gives the same five seeds, to every stored digit, as the three runs that saved everything, so they are one result repeated, not separate evidence. The sensitivity study's run at `fd65a3b-dirty` records a different pair under the same printed settings. It also differs in its machine: it asked for a graphics card and records PyTorch 2.5.1, while the runs at `5602773-dirty` and `f4f40ad` record PyTorch 2.13.0 built for the processor alone (the `+cpu` in its name; such a build cannot use a graphics card), and the four older runs recorded no PyTorch version. Which of the differences moved the result is not established here. `07_comparison_fd001.ipynb` sets the runs side by side seed by seed, because how far one configuration moves between two runs is the yardstick for every comparison that crosses from one run's code and machine to another's.

## Final verification

In [ ]:
checks = [
    ("The window is the folder's window", all(window_checks.values()), f"engine {ENGINE}, cycle {end_cycle}"),
    ("The stage keeps the last column exactly", stage_keeps_last_column, "14 sensors"),
    ("Two different windows give T50 one number", same_number_for_t50,
     f"{abs(pair_out[0] - pair_out[1]):.1e} apart"),
    ("The cost is Table 4's network", all(cost_checks.values()), "0 in the stage, 61,729 in all"),
    ("Each run that saved less repeats one that saved all", unsaved_runs_repeat, f"{len(repeats)} such run(s)"),
]
width = max(len(name) for name, _, _ in checks) + 2
print(f"{'CHECK':<{width}} RESULT   DETAIL")
print("-" * (width + 40))
for name, passed, detail in checks:
    print(f"{name:<{width}} {'PASS' if passed else 'FAIL':<8} {detail}")
print("-" * (width + 40))
failed = [name for name, passed, _ in checks if not passed]
if failed:
    raise AssertionError(f"{len(failed)} check(s) failed: {failed}")
print(f"All {len(checks)} checks passed.")
print(f"\nStored five-seed runs of this stage under this folder's settings: {len(stored)}; the step on stored results lists them, or prints the command that makes one.")

## Summary

### What this notebook produced

- **The stage's output** on one real window: each sensor's reading at the window's last cycle, and nothing else.
- **Two real held-back windows** whose T50 histories differ and which the stage hands on as the same number.
- **Its cost**: no learned numbers in the stage, 61,729 in the network.
- **Its stored results**, each with its commit and its settings.

### Decisions that were not read from the paper

| Decision | Value used | Written up as |
|---|---|---|
| What a sensor's node carries when the recurrent stage is left out | its newest reading | A-07 |
| Leaving the recurrent stage out by default | yes | A-11 |
| No offset in the graph layers | none | A-12 |
| Which window and which sensor are shown | the held-back engine's last window, T50 | `notebooks/temporal/README.md` |

### References

- Wen, J.-R., et al. (2026). Explainable spatiotemporal graph neutral network with dynamic SHAP enhancement for aeroengine RUL prediction. *Aerospace Science and Technology*, 168, 110875. Table 4, Equations 8 to 13 and Table 11.